<!-- course-header -->
# Lesson 21: Deployment, serving and LLMOps

*Worked solutions for lesson 21. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/21-deployment-llmops.html) or [offline](../../lessons/21-deployment-llmops.html). The exercise version without solutions is `../21-deployment-llmops.ipynb`.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/solutions/21-deployment-llmops-solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Take an AI system from notebook to production by building a streaming API service, choosing how to serve open models within a GPU memory budget, controlling latency and cost, and making everything observable: tracing, drift monitoring, cost dashboards, and rollouts gated on evals.

**Runs on:** Any computer (CPU). Running the live service needs an API key and a terminal; every notebook exercise runs offline.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy pandas fastapi "uvicorn[standard]" anthropic httpx

## The anatomy of an LLM application

A production AI feature is mostly ordinary software around a few model calls: an API service (authentication, validation, rate limits), an orchestration layer (prompts, retrieval, tools, routing, guardrails, caching), the model APIs or model servers behind it, and tracing, metrics, logs, evals and feedback around all of it.

Keep model-specific code behind a thin internal interface (`generate(messages, **options)`), keep prompts versioned outside business logic, and make the model name configuration. You will swap models more often than you expect.

## A streaming API with FastAPI

FastAPI is the common choice for Python AI services: typed request validation with Pydantic, async support for many concurrent slow model calls, and automatic API docs.

Run in a terminal:

```bash
uv add fastapi "uvicorn[standard]" anthropic pydantic
```

The next cell saves the service as `app.py` next to this notebook. Run it from a terminal, or import it in the exercise below.

In [ ]:
%%writefile app.py
import json, logging, time, uuid
import anthropic
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

MODEL = "claude-sonnet-5-5"
log = logging.getLogger("app")
logging.basicConfig(level=logging.INFO, format="%(message)s")

app = FastAPI(title="Instrument assistant")
client = anthropic.AsyncAnthropic(max_retries=3, timeout=60)

class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=4000)

def answer_text(message) -> str:
    """The reply may start with a thinking block, so pick the text block rather than content[0]."""
    return "".join(b.text for b in message.content if b.type == "text")

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/ask")
async def ask(req: AskRequest):
    t0 = time.perf_counter()
    r = await client.messages.create(model=MODEL, max_tokens=800,
                                     messages=[{"role": "user", "content": req.question}])
    log.info(json.dumps({"event": "ask", "latency_s": round(time.perf_counter() - t0, 3),
                         "in": r.usage.input_tokens, "out": r.usage.output_tokens}))
    return {"answer": answer_text(r)}

@app.post("/ask/stream")
async def ask_stream(req: AskRequest, request: Request):
    request_id = str(uuid.uuid4())

    async def events():
        t0, first = time.perf_counter(), None
        try:
            async with client.messages.stream(model=MODEL, max_tokens=800,
                                              messages=[{"role": "user", "content": req.question}]) as stream:
                async for text in stream.text_stream:
                    if await request.is_disconnected():          # user left: stop generating and paying
                        return
                    first = first or time.perf_counter()
                    yield f"data: {json.dumps({'text': text})}\n\n"   # server-sent events format
                final = await stream.get_final_message()
            yield f"data: {json.dumps({'done': True})}\n\n"
            log.info(json.dumps({"event": "ask_stream", "id": request_id,
                                 "ttft_s": round((first or time.perf_counter()) - t0, 3),
                                 "total_s": round(time.perf_counter() - t0, 3),
                                 "in": final.usage.input_tokens, "out": final.usage.output_tokens}))
        except anthropic.APIError as e:
            log.error(json.dumps({"event": "upstream_error", "id": request_id, "error": repr(e)}))
            yield f"data: {json.dumps({'error': 'The model service is busy. Please try again.'})}\n\n"

    return StreamingResponse(events(), media_type="text/event-stream")

Run in a terminal:

```bash
uv run uvicorn app:app --reload
curl -N -X POST localhost:8000/ask/stream -H "Content-Type: application/json" \
     -d '{"question": "Explain the KV cache in three sentences."}'
```

Visit `localhost:8000/docs` for the interactive documentation FastAPI generates.

Two details are easy to miss. The SDK's `text_stream` yields only text, but a full reply is a list of content blocks that may begin with a thinking block, so never read `content[0].text`; `answer_text` picks the text blocks. And the stream loop checks whether the client has gone away, because tokens you generate for a closed connection still cost money.

### Ship it in a container

*Dockerfile*

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY pyproject.toml uv.lock ./
RUN pip install uv && uv sync --frozen --no-dev
COPY . .                       # list .venv and .env in .dockerignore so they are not copied
EXPOSE 8000
CMD ["uv", "run", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
```

Secrets come from the environment at runtime (`docker run -e ANTHROPIC_API_KEY=...`, or your platform's secret manager), never baked into the image.

### Hands-on: test the service with a fake model

A service you can only test with a live API key and a credit card will not get tested. FastAPI's `TestClient` runs the app in-process, and the model client is an attribute you can replace.

1. Write a `FakeMessages` class with an async `create` that returns an object with `content` (one text block) and `usage`, and a `stream` method that returns an async context manager with a `text_stream` and `get_final_message`.
2. Import `app`, replace `app.client.messages` with your fake, and use `TestClient` to check: `/health` returns 200; `/ask` with an empty question returns 422; `/ask` returns the fake answer; `/ask/stream` returns `text/event-stream` whose last event is `{"done": true}`.
3. Make the fake raise `anthropic.APIConnectionError` and check the stream reports an error event instead of crashing.

You are done when all five checks pass without a network connection. The `%%writefile` cell above saved `app.py` next to this notebook, so `import app` works here.

#### Solution

In [ ]:
import contextlib, json, os
from types import SimpleNamespace
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key")      # the client is built at import time
import anthropic
from fastapi.testclient import TestClient
import app as service

def fake_message(text="The KV cache stores keys and values."):
    return SimpleNamespace(content=[SimpleNamespace(type="text", text=text)],
                           usage=SimpleNamespace(input_tokens=12, output_tokens=8))

class FakeMessages:
    def __init__(self, fail=False):
        self.fail = fail

    async def create(self, **kw):
        return fake_message()

    def stream(self, **kw):
        @contextlib.asynccontextmanager
        async def cm():
            if self.fail:
                raise anthropic.APIConnectionError(request=None)
            async def words():
                for w in ["The ", "KV ", "cache."]:
                    yield w
            yield SimpleNamespace(text_stream=words(), get_final_message=_final)
        return cm()

async def _final():
    return fake_message()

service.client.messages = FakeMessages()
tc = TestClient(service.app)
assert tc.get("/health").status_code == 200
assert tc.post("/ask", json={"question": ""}).status_code == 422
assert tc.post("/ask", json={"question": "hi"}).json()["answer"].startswith("The KV cache")
r = tc.post("/ask/stream", json={"question": "hi"})
events = [json.loads(line[6:]) for line in r.text.splitlines() if line.startswith("data: ")]
assert r.headers["content-type"].startswith("text/event-stream") and events[-1] == {"done": True}
service.client.messages = FakeMessages(fail=True)
r = tc.post("/ask/stream", json={"question": "hi"})
assert "error" in r.text
print("all checks passed:", len(events), "events on the happy path")

The fake mirrors only the three things the app touches: `content`, `usage` and the stream protocol. That is the point of keeping the model behind a thin interface; a fake for a client that was called from twelve places would be a project in itself.

The error test matters most. A streaming endpoint that throws after the headers were sent cannot return a 500, so the error has to travel inside the stream.

## Serving open models

### The serving landscape

Run in a terminal:

```bash
# vLLM: an OpenAI-compatible server on port 8000
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 16384 --gpu-memory-utilization 0.9

# serve a base model plus your LoRA adapter from lesson 18
vllm serve Qwen/Qwen2.5-0.5B-Instruct --enable-lora --lora-modules notes=./notes-to-json-lora
```

### The GPU memory budget

In [ ]:
def serving_memory_gb(params_billion, weight_bytes, layers, kv_heads, head_dim,
                      context_tokens, concurrent_seqs, kv_bytes=2):
    weights = params_billion * 1e9 * weight_bytes
    kv_per_token = 2 * layers * kv_heads * head_dim * kv_bytes      # keys and values, every layer
    kv_total = kv_per_token * context_tokens * concurrent_seqs
    return round(weights / 1e9, 1), round(kv_total / 1e9, 1)

# an 8B model with 32 layers, 8 KV heads of dimension 128:
print(serving_memory_gb(8, 2, 32, 8, 128, 8192, 16))     # (16.0, 17.2): BF16 weights + 16 users at 8k context
print(serving_memory_gb(8, 0.5, 32, 8, 128, 8192, 16))   # (4.0, 17.2): 4-bit weights; KV cache now dominates

### Hands-on: capacity-plan a GPU

You must serve the 8B model above (32 layers, 8 KV heads of dimension 128) to users whose conversations reach 8,192 tokens.

1. Write `max_concurrent(gpu_gb, weight_bytes, context_tokens, kv_bytes=2)` that returns how many sequences fit after the weights, keeping 15% of the card free for activations and overheads.
2. Print a table for 24 GB, 48 GB and 80 GB cards at BF16, INT8 and 4-bit weights.
3. Add two more rows for the 80 GB card: KV cache quantised to 8-bit (`kv_bytes=1`), and a 4,096-token context limit.

You are done when you can name the cheapest configuration that serves 32 users at 8k context, and explain why quantising the weights alone does not get a 24 GB card there.

#### Solution

In [ ]:
def max_concurrent(gpu_gb, weight_bytes, context_tokens, kv_bytes=2,
                   params_billion=8, layers=32, kv_heads=8, head_dim=128, headroom=0.15):
    weights_gb = params_billion * weight_bytes
    kv_per_seq_gb = 2 * layers * kv_heads * head_dim * kv_bytes * context_tokens / 1e9
    free = gpu_gb * (1 - headroom) - weights_gb
    return max(0, int(free // kv_per_seq_gb))

print(f"{'GPU':>6} {'weights':>8} {'context':>8} {'kv':>3}  users")
for gpu in [24, 48, 80]:
    for name, wb in [("bf16", 2), ("int8", 1), ("4-bit", 0.5)]:
        print(f"{gpu:>4}GB {name:>8} {8192:>8} {2:>3}  {max_concurrent(gpu, wb, 8192)}")
print(f"{80:>4}GB {'4-bit':>8} {8192:>8} {1:>3}  {max_concurrent(80, 0.5, 8192, kv_bytes=1)}")
print(f"{80:>4}GB {'4-bit':>8} {4096:>8} {2:>3}  {max_concurrent(80, 0.5, 4096)}")

Each 8k-token sequence costs about 1.07 GB of KV cache. A 24 GB card with 4-bit weights has about 16 GB left, so 15 users, however small the weights get: the cache, not the weights, is the wall.

An 80 GB card holds about 48 users at BF16 and about 59 at 4-bit. Quantising the KV cache to 8-bit doubles that, and so does halving the context limit. The cheapest configuration that serves 32 users at 8k is a single 48 GB card with 4-bit weights (34 users). INT8 weights fall just short at 30, and KV-cache quantisation is the next lever.

Real servers reserve memory differently, so treat the numbers as a planning estimate and confirm with a load test.

## Latency, throughput and cost

### Levers, roughly in order of payoff

In [ ]:
import numpy as np

class SemanticCache:
    def __init__(self, embed, threshold=0.95):
        self.embed, self.threshold = embed, threshold
        self.vecs, self.answers = [], []

    def get(self, question: str):
        if not self.vecs:
            return None
        q = self.embed(question)
        sims = np.array(self.vecs) @ q
        i = int(sims.argmax())
        return self.answers[i] if sims[i] >= self.threshold else None

    def put(self, question: str, answer: str):
        self.vecs.append(self.embed(question)); self.answers.append(answer)

In [ ]:
import re, hashlib

def toy_embed(text, dim=256):
    """Hashed bag of words: a stand-in for a real embedding model, good enough to show the mechanics."""
    v = np.zeros(dim)
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1
    return v / (np.linalg.norm(v) or 1)

cache = SemanticCache(toy_embed, threshold=0.8)
cache.put("How do I replace the desiccant?", "Open the enclosure, swap the cartridge, check the indicator is blue.")
for q in ["how do I replace the desiccant", "How do I replace the filter wheel belt?"]:
    print(f"{q!r:<45} -> {cache.get(q)}")

### A cost dashboard from your logs

The JSON log lines the service writes are the raw material for every cost question. A **cost dashboard** needs only four things per request: a timestamp, a feature name, input and output token counts, and the model. Cost follows by multiplying by the price list you keep in configuration.

Aggregate by day and feature, by user for quota alerts, and by model to see what a routing change saved. Add the cache hit rate: every hit is a request that cost nothing. The point is not the chart but the habit: the person who changed the prompt sees the cost line move the next morning.

### Hands-on: a cost and latency dashboard from logs

1. Generate 2,000 synthetic log lines like the ones `app.py` writes, spread over 7 days and three features (`ask`, `summarise`, `extract`), with realistic token counts, a cache-hit flag, and a `model` field that routes 30% of `ask` traffic to a cheaper model. Make one day's `summarise` prompts twice as long as the others.
2. Load the lines with pandas, compute the cost of each request from a price table (per million input and output tokens), and print cost per feature per day and the cache hit rate.
3. Print p50 and p95 latency per feature.

You are done when the day with the longer prompts stands out in the table and you can say what the routing to the cheaper model saved.

#### Solution

In [ ]:
import json, random
import pandas as pd

random.seed(0)
PRICES = {"big": (2.0, 10.0), "small": (0.10, 0.50)}         # dollars per million input / output tokens
rows = []
for i in range(2000):
    day = i % 7
    feature = random.choice(["ask", "ask", "summarise", "extract"])
    model = "small" if feature == "ask" and random.random() < 0.3 else "big"
    inp = {"ask": 900, "summarise": 3000, "extract": 1500}[feature] * (2 if feature == "summarise" and day == 4 else 1)
    inp = int(random.gauss(inp, inp * 0.2)); out = int(random.gauss(250, 80))
    hit = feature == "ask" and random.random() < 0.25
    rows.append({"ts": f"2026-10-{12 + day:02d}", "feature": feature, "model": model,
                 "in": 0 if hit else inp, "out": 0 if hit else out, "cache_hit": hit,
                 "latency_s": 0.05 if hit else round(0.4 + out * (0.03 if model == "big" else 0.015) + random.random(), 2)})
with open("requests.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

df = pd.read_json("requests.jsonl", lines=True)
df["cost"] = [(PRICES[m][0] * i + PRICES[m][1] * o) / 1e6 for m, i, o in zip(df.model, df["in"], df["out"])]
print(df.pivot_table(index="ts", columns="feature", values="cost", aggfunc="sum").round(2))
print("cache hit rate:", round(df.cache_hit.mean(), 3))
print(df.groupby("feature").latency_s.quantile([0.5, 0.95]).unstack().round(2))
ask = df[(df.feature == "ask") & ~df.cache_hit]
would_cost = (PRICES["big"][0] * ask["in"] + PRICES["big"][1] * ask["out"]).sum() / 1e6
print(f"ask: actual ${ask.cost.sum():.2f} versus ${would_cost:.2f} with no routing")

The `summarise` column jumps on 2026-10-16, the day whose prompts are twice as long. That is exactly the kind of silent prompt growth that a dashboard catches and a unit test never will.

Routing 30% of the easy questions to the small model cuts the `ask` bill by about a quarter. Whether that is worth it depends on the eval, not the dashboard. In production the same three queries run against a log store or a metrics system instead of a file, and the price table lives in configuration next to the model name.

## Observability

### Tracing

Tools: OpenTelemetry with its generative-AI conventions for vendor-neutral tracing; LLM-specific platforms such as Langfuse (open source), Arize Phoenix, LangSmith, Helicone or Weights & Biases Weave, which add prompt management, cost tracking, datasets and evals on top of traces.

In [ ]:
# a minimal hand-rolled span recorder; production code would use OpenTelemetry or a tracing SDK
import contextlib, json, time, uuid

@contextlib.contextmanager
def span(name, trace_id, **attrs):
    t0 = time.perf_counter()
    record = {"trace": trace_id, "span": name, **attrs}
    try:
        yield record                      # callers add outputs, token counts, etc.
    except Exception as e:
        record["error"] = repr(e); raise
    finally:
        record["ms"] = round(1000 * (time.perf_counter() - t0), 1)
        print(json.dumps(record))         # ship to your log pipeline

trace_id = str(uuid.uuid4())
with span("retrieve", trace_id, query="belt slipping") as s:
    s["chunk_ids"] = ["manual-x#12", "manual-x#13"]

> **Traces contain user data**
>
> Prompts and outputs often include personal or confidential information. Decide what to store, redact where needed, restrict access, and set retention periods. Privacy obligations apply to logs too.

### Monitoring for drift

**Drift** is change without error. Three kinds matter for LLM systems:

- **Input drift**: users ask about new products, in new languages, with longer messages, or a new client sends a different kind of traffic.
- **Output drift**: answer length, refusal rate, tool-use rate and judge scores shift, usually after a provider model update, an unevaluated prompt change or an index rebuild.
- **Quality drift**: the offline eval set still passes but online feedback worsens, so the eval set no longer represents the traffic.

A practical detector compares this week's distribution of a metric with a reference week. For categorical metrics a chi-square test or the **population stability index** (PSI) works; for numeric ones, compare percentiles. Alert on the comparison, not on the raw number, so that normal daily variation does not page anyone.

### Hands-on: detect drift in production traffic

1. Simulate two weeks of request-type counts for the support assistant: a reference week where requests are 50% hardware, 20% software, 20% data and 10% other, and a second week with the same mix. Then simulate a third week in which a new client arrives and data questions rise to 40%.
2. Implement the population stability index, $\text{PSI} = \sum_i (p_i - q_i)\ln(p_i / q_i)$ over categories, and compute it for weeks two and three against the reference week.
3. Also track the mean input length per day for the three weeks, with a jump in week three, and flag any day whose mean is more than three standard deviations from the reference week.

You are done when week two stays under the common PSI alert threshold of 0.2, week three exceeds it, and the length alert fires only in week three.

#### Solution

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
CATS = ["hardware", "software", "data", "other"]

def week(mix, n=1400):
    counts = rng.multinomial(n, mix)
    return counts / counts.sum()

def psi(p, q, eps=1e-6):
    p, q = np.asarray(p) + eps, np.asarray(q) + eps
    return float(np.sum((p - q) * np.log(p / q)))

ref = week([0.5, 0.2, 0.2, 0.1])
for name, mix in [("week 2", [0.5, 0.2, 0.2, 0.1]), ("week 3", [0.35, 0.15, 0.4, 0.1])]:
    value = psi(week(mix), ref)
    print(f"{name}: PSI {value:.3f} {'ALERT' if value > 0.2 else 'ok'}")

ref_days = rng.normal(180, 12, 7)                       # mean input length per day, reference week
mu, sd = ref_days.mean(), ref_days.std()
for name, mean in [("week 2", 180), ("week 3", 260)]:
    days = rng.normal(mean, 12, 7)
    flags = [f"day {i + 1}" for i, d in enumerate(days) if abs(d - mu) > 3 * sd]
    print(f"{name}: mean length {days.mean():.0f}, alerts: {flags or 'none'}")

Week two lands near zero: sampling noise alone gives a PSI of a few thousandths. Week three lands above 0.2 because a fifth of the traffic moved category. The exact PSI for this shift is about 0.21, so with another seed the sample can fall either side of the line; a bigger shift clears it easily.

The length detector is the same idea for a numeric metric. Two refinements for production: compare against a rolling reference so that slow seasonal change does not accumulate into a false alert, and when an alert fires, pull a sample of the new traffic into your eval set before anyone touches the prompt.

### Rolling out changes: canaries, A/B tests and shadow deployments

Every prompt, model or index change is a deployment. A **canary** sends 1 to 5% of traffic to the new version; an **A/B test** splits users and compares the outcome the product exists for; a **shadow deployment** runs the candidate on live traffic but shows nobody its answers. The versions must be distinguishable in traces and logs, and rollback must be a configuration change, not a code deploy.

### CI/CD for AI systems

Version everything (code, prompts, model identifiers, index builds, eval sets), gate deploys on evals, roll out gradually with automatic rollback, and shadow first for model swaps. The exercise below puts the whole lesson together.

### Hands-on: production-ise your RAG system

1. Wrap your lesson 14 `MiniRAG` in a FastAPI service with `/ask` and `/ask/stream`, validating input with Pydantic.
2. Log one JSON line per request with: request id, retrieved chunk ids, model, input and output tokens, TTFT, total latency and estimated cost.
3. Add an exact-match response cache, then a semantic cache, and measure hit rates on a replay of 100 questions (include paraphrases).
4. Write a load test (for example with `asyncio` firing 50 concurrent requests, or the Locust tool) and report p50 and p95 latency.
5. Containerise it and run it with `docker run`.

#### Solution: what to measure and typical findings

The load test below runs against the service in `app.py` in-process, through `httpx.ASGITransport`, with a fake model that sleeps like a real provider. No server, key or network is needed. Wrap your own RAG service the same way, or drop the transport to hit a server you started with `uv run uvicorn app:app`.

In [ ]:
import asyncio, logging, os, random, time, httpx, numpy as np
from types import SimpleNamespace
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key")      # the client is built at import time
import app as service

async def one(client, q):
    t0 = time.perf_counter()
    r = await client.post("http://localhost:8000/ask", json={"question": q}, timeout=120)
    return time.perf_counter() - t0, r.status_code

async def load_test(questions, concurrency=50, transport=None):
    async with httpx.AsyncClient(transport=transport) as client:   # no transport: a real network call
        res = await asyncio.gather(*(one(client, q) for q in questions[:concurrency]))
    lat = np.array([t for t, code in res if code == 200])
    print(f"ok {len(lat)}/{len(res)}  p50 {np.percentile(lat, 50):.2f}s  p95 {np.percentile(lat, 95):.2f}s")

class SlowFakeMessages:
    """A fake model with provider-like latency: most replies are quick, a few are slow."""
    def __init__(self, seed=0):
        self.rng = random.Random(seed)

    async def create(self, **kw):
        await asyncio.sleep(min(self.rng.lognormvariate(-1.5, 0.6), 2.0))
        return SimpleNamespace(content=[SimpleNamespace(type="text", text="A fake answer.")],
                               usage=SimpleNamespace(input_tokens=900, output_tokens=120))

for name in ("app", "httpx"):                    # 100 log lines would bury the result
    logging.getLogger(name).setLevel(logging.WARNING)
service.client.messages = SlowFakeMessages()
questions = [f"Question {i}: why is the filter wheel belt slipping?" for i in range(50)]
# A notebook already runs an event loop, so await directly; in a script use asyncio.run(...).
await load_test(questions, transport=httpx.ASGITransport(app=service.app))

# Against your running server: await load_test(my_questions)

Typical findings: p95 is often two to three times p50, driven by long answers and provider-side queueing; a few requests may hit rate limits under burst load, which your retries should absorb (check the logs). The semantic cache catches paraphrases that the exact cache misses, but with a loose threshold it also returns answers to subtly different questions; reading those false hits is the best way to choose the threshold. The JSON logs make all of this answerable in a few lines of pandas (lesson 01 skills, paying off).

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/21-deployment-llmops.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.